# CodeTune v2 — HumanEval Evaluation
Self-contained. No file uploads needed.

**Models**: `base` · `sft_A` · `sft_C`

## 0 · Configuration — fill in before running

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
HF_TOKEN = "YOUR_HF_TOKEN"

BASE_MODEL  = "unsloth/Qwen3.5-9B"
SFT_A_MODEL = "Michlitt/codetune-v2-sft-A"
SFT_C_MODEL = "Michlitt/codetune-v2-sft-C"
DPO_MODEL   = "Michlitt/codetune-v2-dpo"   # dpo_v2 uploaded here

SFT_IS_ADAPTER = True

MAX_SEQ_LEN    = 2048
MAX_NEW_TOKENS = 512
LOAD_IN_4BIT   = True   # L4 22GB — bf16 Qwen3.5-9B ~18GB too tight

EXEC_TIMEOUT = 10
SAVE_EVERY   = 10

DRIVE_RESULTS = "/content/drive/MyDrive/codetune/eval_results"
# ────────────────────────────────────────────────────────────────────────────

MODELS = [
    (BASE_MODEL,  "base",   False),
    (SFT_A_MODEL, "sft_A",  SFT_IS_ADAPTER),
    (SFT_C_MODEL, "sft_C",  SFT_IS_ADAPTER),
    (DPO_MODEL,   "dpo_v1", SFT_IS_ADAPTER),   # old checkpoint — skip if already done
    (DPO_MODEL,   "dpo_v2", SFT_IS_ADAPTER),   # new 450-pair checkpoint
]

print("Models to evaluate:")
for m, lbl, adpt in MODELS:
    print(f"  [{lbl}]  {m}  {'(adapter)' if adpt else '(full)'}")
print(f"\nLOAD_IN_4BIT = {LOAD_IN_4BIT}")

## 1 · GPU check + install

In [2]:
import subprocess, torch
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()
print("GPU  :", gpu)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("BF16 :", torch.cuda.is_bf16_supported())

GPU  : NVIDIA L4, 23034 MiB, 22564 MiB
Torch: 2.10.0+cu128 | CUDA: 12.8
BF16 : True


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"]          = "true"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install peft accelerate bitsandbytes datasets huggingface-hub tqdm -q
print("Done.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 38.5 MB/s eta 0:00:00
Done.


## 2 · HF login + Drive mount

In [4]:
import os
from huggingface_hub import login

assert HF_TOKEN, "Set HF_TOKEN in the Config cell!"
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    print("Drive mounted →", DRIVE_RESULTS)
except Exception as e:
    DRIVE_RESULTS = "/content/eval_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    print(f"Drive unavailable ({e}). Saving to {DRIVE_RESULTS}")

HF login OK
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted → /content/drive/MyDrive/codetune/eval_results


## 3 · Core functions

In [ ]:
from __future__ import annotations
import gc, json, re, subprocess, sys, tempfile
from pathlib import Path
import torch
from tqdm import tqdm

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Complete the given function. "
    "Write only the function body — no extra explanation, no markdown fences, "
    "no test code."
)

TARGETED_TASKS = {
    "HumanEval/54":  "same_chars",
    "HumanEval/55":  "fib",
    "HumanEval/107": "minSubArraySum",
    "HumanEval/108": "intersection",
    "HumanEval/110": "prod_signs",
    "HumanEval/128": "tri",
    "HumanEval/141": "file_name_check",
    "HumanEval/147": "get_max_triples",
}


# ── Completion cleanup ────────────────────────────────────────────────
def clean_completion(raw: str, prompt: str) -> str:
    # Strip only trailing whitespace; preserve leading indent on first line
    text = raw.rstrip()

    # Strip Qwen3 think blocks
    text = re.sub(r"<think>[\s\S]*?</think>\s*", "", text)
    # Strip markdown fences
    text = re.sub(r"^```(?:python)?\s*\n", "", text)
    text = re.sub(r"\n?```\s*$", "", text)
    text = text.strip("\n")  # only strip newlines, not spaces

    # If model echoed the function header + docstring, strip them
    fn_match = re.search(r"^def\s+(\w+)", prompt, re.MULTILINE)
    if fn_match:
        fn = re.escape(fn_match.group(1))
        hdr = re.search(rf"def\s+{fn}\s*\(.*?\).*?:\s*\n", text, re.DOTALL)
        if hdr:
            after = text[hdr.end():]
            doc = re.match(r'\s*"""[\s\S]*?"""\s*\n', after)
            text = after[doc.end():] if doc else after
    text = text.strip("\n")

    # Normalize indentation to 4-space base.
    # Compute min indent across all non-empty lines (preserving leading spaces),
    # then shift so that the base becomes exactly 4.
    lines = text.splitlines()
    non_empty = [l for l in lines if l.strip()]
    if not non_empty:
        return text
    min_indent = min(len(l) - len(l.lstrip()) for l in non_empty)
    if min_indent != 4:
        shift = 4 - min_indent
        result = []
        for l in lines:
            if not l.strip():
                result.append("")
            elif shift > 0:
                result.append(" " * shift + l)
            else:
                result.append(l[-shift:])
        text = "\n".join(result)

    return text


# ── Model loading ─────────────────────────────────────────────────────
def load_model(model_id: str, is_adapter: bool):
    from unsloth import FastLanguageModel
    dtype = torch.bfloat16 if not LOAD_IN_4BIT else None
    if is_adapter:
        print(f"  Base  : {BASE_MODEL}")
        print(f"  Adapter: {model_id}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=LOAD_IN_4BIT, dtype=dtype,
        )
    else:
        print(f"  Model: {model_id}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id, max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=LOAD_IN_4BIT, dtype=dtype,
        )
    FastLanguageModel.for_inference(model)
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"  Model loaded. VRAM: {vram:.1f} GB")
    return model, tokenizer

def unload_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"  Model unloaded. VRAM in use: {vram:.1f} GB")


# ── Generation ────────────────────────────────────────────────────────
DEBUG_RAW = False

def generate(model, tokenizer, prompt: str, _debug_tid: str = "") -> str:
    _tok = getattr(tokenizer, "tokenizer", tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Complete this Python function:\n\n{prompt}"},
    ]
    text = _tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    text = text + "<think>\n\n</think>\n\n"

    inputs = _tok(
        text=text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN,
    ).to(model.device)

    stop_ids = [_tok.eos_token_id]
    im_end = _tok.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end > 0:
        stop_ids.append(im_end)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            eos_token_id=stop_ids,
            pad_token_id=_tok.eos_token_id,
        )

    raw = _tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    if DEBUG_RAW and _debug_tid:
        print(f"\n[DEBUG raw {_debug_tid}]: {raw!r}\n")

    return clean_completion(raw, prompt)


# ── Test execution ────────────────────────────────────────────────────
def run_tests(prompt: str, completion: str, test_code: str, entry_point: str) -> dict:
    full = prompt + completion + "\n\n" + test_code + f"\n\ncheck({entry_point})\n"
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8"
    ) as f:
        f.write(full); tmp = f.name
    try:
        r = subprocess.run(
            [sys.executable, tmp], capture_output=True, text=True, timeout=EXEC_TIMEOUT,
        )
        return {"passed": r.returncode == 0, "stderr": r.stderr[:400]}
    except subprocess.TimeoutExpired:
        return {"passed": False, "stderr": "TIMEOUT"}
    except Exception as e:
        return {"passed": False, "stderr": str(e)}
    finally:
        Path(tmp).unlink(missing_ok=True)


# ── Checkpoint save ───────────────────────────────────────────────────
def _save_checkpoint(label: str, total: int, per_problem: dict, out_path: Path) -> dict:
    passed = sum(1 for v in per_problem.values() if v["passed"])
    summary = {
        "label": label, "total": total, "passed": passed,
        "pass_at_1": passed / total if total else 0.0,
        "failed_ids": [tid for tid, v in per_problem.items() if not v["passed"]],
        "per_problem": per_problem,
    }
    out_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return summary


# ── Full evaluation ───────────────────────────────────────────────────
def evaluate(model, tokenizer, label: str, problems: list) -> dict:
    out_path = Path(DRIVE_RESULTS) / f"humaneval_{label}.json"

    per_problem = {}
    if out_path.exists():
        try:
            per_problem = json.loads(out_path.read_text(encoding="utf-8")).get("per_problem", {})
            print(f"  Resuming: {len(per_problem)}/{len(problems)} already done.")
        except Exception:
            pass

    remaining = [p for p in problems if p["task_id"] not in per_problem]
    total_done_start = len(per_problem)

    for i, prob in enumerate(tqdm(remaining, desc=f"[{label}]", unit="prob")):
        task_id    = prob["task_id"]
        debug_tid  = task_id if (DEBUG_RAW and i < 3) else ""
        completion = generate(model, tokenizer, prob["prompt"], _debug_tid=debug_tid)
        tr         = run_tests(prob["prompt"], completion, prob["test"], prob["entry_point"])

        per_problem[task_id] = {
            "passed":     tr["passed"],
            "completion": completion[:700],
            "stderr":     tr["stderr"],
        }

        n_done = total_done_start + i + 1
        flag   = "✓" if tr["passed"] else "✗"
        note   = " ← targeted" if task_id in TARGETED_TASKS else ""
        err    = f"  {tr['stderr'][:80]}" if not tr["passed"] and tr["stderr"] else ""
        print(f"  [{n_done:>3}/{len(problems)}] {flag} {task_id}{note}{err}")

        if (i + 1) % SAVE_EVERY == 0:
            _save_checkpoint(label, len(problems), per_problem, out_path)
            passed_so_far = sum(1 for v in per_problem.values() if v["passed"])
            print(f"    [checkpoint — {passed_so_far}/{n_done} passed]")

    summary = _save_checkpoint(label, len(problems), per_problem, out_path)
    p, t = summary["passed"], summary["total"]
    print(f"\n  ► {label}: {p}/{t} = {p/t:.1%}  pass@1")
    print(f"  ► Failed: {summary['failed_ids']}\n")
    return summary


print("Core functions loaded.")

## 4 · Load HumanEval dataset

In [6]:
from datasets import load_dataset
he_ds = load_dataset("openai/openai_humaneval", split="test", trust_remote_code=True)
HE_PROBLEMS = list(he_ds)
print(f"Loaded {len(HE_PROBLEMS)} problems.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/openai_humaneval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/openai_humaneval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be 

Loaded 164 problems.


## 5 · Sanity check — 1 problem with sft_C
Confirm generation and test execution work before running all 164.

In [7]:
# ── accelerate 兼容补丁（Transformers 5.x _no_split_modules 改成了 set）──
import accelerate.utils.modeling as _am

_orig_gbm = _am.get_balanced_memory

def _fixed_gbm(model, max_memory=None, no_split_module_classes=None, **kwargs):
    if no_split_module_classes is not None:
        flat = []
        for c in no_split_module_classes:
            if isinstance(c, (set, frozenset)):
                flat.extend(str(x) for x in c)
            elif isinstance(c, type):
                flat.append(c.__name__)
            else:
                flat.append(str(c))
        no_split_module_classes = flat
    return _orig_gbm(
        model,
        max_memory=max_memory,
        no_split_module_classes=no_split_module_classes,
        **kwargs,
    )

_am.get_balanced_memory = _fixed_gbm
print("accelerate patched ✓")

accelerate patched ✓


In [8]:
print("Loading sft_C for sanity check...")
_m, _tok = load_model(SFT_C_MODEL, is_adapter=SFT_IS_ADAPTER)

prob = HE_PROBLEMS[0]
comp = generate(_m, _tok, prob["prompt"])
res  = run_tests(prob["prompt"], comp, prob["test"], prob["entry_point"])

print("=== Prompt ===")
print(prob["prompt"])
print("\n=== Completion ===")
print(comp)
print(f"\n=== Result: {'PASS ✓' if res['passed'] else 'FAIL ✗'} ===")
if not res["passed"]:
    print("stderr:", res["stderr"])

# Free memory before main eval
unload_model(_m, _tok)
del _m, _tok

Loading sft_C for sanity check...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  Base  : unsloth/Qwen3.5-9B
  Adapter: Michlitt/codetune-v2-sft-C
==((====))==  Unsloth 2026.3.5: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

=== Prompt ===
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


=== Completion ===
    numbers.sort()
    for i in range(len(numbers) - 1):
        if abs(numbers[i] - numbers[i + 1]) < threshold:
            return True
    return False

=== Result: PASS ✓ ===
  Model unloaded. VRAM in use: 19.3 GB


## 6 · Main evaluation — base · sft_A · sft_C
Loads each model in turn, runs 164 HumanEval problems, unloads before next model.
Saves checkpoint every 10 problems. Safe to re-run if interrupted — resumes from checkpoint.

In [12]:
all_summaries = {}

for model_id, label, is_adapter in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"{'='*60}")

    model, tokenizer = load_model(model_id, is_adapter)
    all_summaries[label] = evaluate(model, tokenizer, label, HE_PROBLEMS)
    unload_model(model, tokenizer)
    del model, tokenizer

print("\nAll evaluations complete.")


Evaluating: sft_C
  Base  : unsloth/Qwen3.5-9B
  Adapter: Michlitt/codetune-v2-sft-C
==((====))==  Unsloth 2026.3.5: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

[sft_C]:   1%|          | 1/164 [00:07<21:40,  7.98s/prob]

  [  1/164] ✓ HumanEval/0


[sft_C]:   1%|          | 2/164 [00:21<30:33, 11.32s/prob]

...

[sft_C]:  99%|█████████▉| 163/164 [29:04<00:16, 16.94s/prob]

  [163/164] ✓ HumanEval/162


[sft_C]: 100%|██████████| 164/164 [29:14<00:00, 10.70s/prob]

  [164/164] ✗ HumanEval/163  Traceback (most recent call last):
  File "/tmp/tmpup7f2lac.py", line 32, in <mo

  ► sft_C: 125/164 = 76.2%  pass@1
  ► Failed: ['HumanEval/9', 'HumanEval/10', 'HumanEval/26', 'HumanEval/32', 'HumanEval/38', 'HumanEval/50', 'HumanEval/54', 'HumanEval/64', 'HumanEval/67', 'HumanEval/75', 'HumanEval/77', 'HumanEval/81', 'HumanEval/83', 'HumanEval/86', 'HumanEval/91', 'HumanEval/93', 'HumanEval/95', 'HumanEval/96', 'HumanEval/106', 'HumanEval/108', 'HumanEval/109', 'HumanEval/115', 'HumanEval/120', 'HumanEval/126', 'HumanEval/128', 'HumanEval/129', 'HumanEval/130', 'HumanEval/131', 'HumanEval/132', 'HumanEval/133', 'HumanEval/134', 'HumanEval/135', 'HumanEval/137', 'HumanEval/138', 'HumanEval/142', 'HumanEval/145', 'HumanEval/159', 'HumanEval/160', 'HumanEval/163']



  Model unloaded. VRAM in use: 20.7 GB

All evaluations complete.


## 7 · Results summary

In [16]:
import json
from pathlib import Path

# Load all saved results (works even if cell 6 was split across sessions)
all_data = {}
for model_id, label, _ in MODELS:
    fp = Path(DRIVE_RESULTS) / f"humaneval_{label}.json"
    if fp.exists():
        all_data[label] = json.loads(fp.read_text(encoding="utf-8"))
    else:
        print(f"WARNING: {fp} not found — run cell 6 first.")

labels = [lbl for _, lbl, _ in MODELS if lbl in all_data]

# ── Overall pass@1 ───────────────────────────────────────────────────
print(f"\n{'Model':<10} {'pass@1':>8} {'passed':>12} {'failed':>8}")
print("-" * 42)
for lbl in labels:
    d = all_data[lbl]
    print(f"{lbl:<10} {d['pass_at_1']:>8.1%} "
          f"{d['passed']:>5}/{d['total']:<5} {d['total']-d['passed']:>6}")

# ── Targeted tasks ───────────────────────────────────────────────────
col = 8
w   = 28 + col * len(labels)
print(f"\n\nTargeted failure cases  (Coder-Agent baseline: 0/8)")
print("=" * w)
print(f"{'Task':<28}" + "".join(f"{l:>{col}}" for l in labels))
print("-" * w)
for tid, fn in TARGETED_TASKS.items():
    row = f"{tid + ' (' + fn + ')':<28}"
    for lbl in labels:
        v = all_data[lbl].get("per_problem", {}).get(tid, {}).get("passed", None)
        row += f"{'✓' if v else ('✗' if v is False else '?'):>{col}}"
    print(row)
print("-" * w)
totals = f"{'TARGETED TOTAL':<28}"
for lbl in labels:
    per = all_data[lbl].get("per_problem", {})
    n = sum(1 for tid in TARGETED_TASKS if per.get(tid, {}).get("passed"))
    totals += f"{f'{n}/8':>{col}}"
print(totals)
print("=" * w)


Model        pass@1       passed   failed
------------------------------------------
base          37.8%    62/164      102
sft_A         65.2%   107/164       57
sft_C         76.2%   125/164       39


Targeted failure cases  (Coder-Agent baseline: 0/8)
Task                            base   sft_A   sft_C
----------------------------------------------------
HumanEval/54 (same_chars)          ✗       ✗       ✗
HumanEval/55 (fib)                 ✗       ✓       ✓
HumanEval/107 (minSubArraySum)       ✓       ✓       ✓
HumanEval/108 (intersection)       ✗       ✗       ✗
HumanEval/110 (prod_signs)         ✗       ✗       ✓
HumanEval/128 (tri)                ✗       ✓       ✗
HumanEval/141 (file_name_check)       ✗       ✗       ✓
HumanEval/147 (get_max_triples)       ✓       ✓       ✓
----------------------------------------------------
TARGETED TOTAL                   2/8     4/8     5/8


## 8 · (Debug) Inspect specific failures

In [18]:
INSPECT_LABEL   = "sft_C"          # base / sft_A / sft_C
INSPECT_TASK_ID = None   # or None to show first N failures
MAX_SHOW        = 8

fp = Path(DRIVE_RESULTS) / f"humaneval_{INSPECT_LABEL}.json"
d  = json.loads(fp.read_text(encoding="utf-8"))
per = d["per_problem"]

tasks = [INSPECT_TASK_ID] if INSPECT_TASK_ID else [
    tid for tid, v in per.items() if not v["passed"]
][:MAX_SHOW]

for tid in tasks:
    v = per.get(tid, {})
    print(f"\n{'='*60}")
    print(f"{tid}  —  {'PASS ✓' if v.get('passed') else 'FAIL ✗'}")
    print(f"Completion:\n{v.get('completion', '(none)')}")
    if not v.get("passed"):
        print(f"stderr: {v.get('stderr', '')}")


HumanEval/9  —  FAIL ✗
Completion:
    max_num = numbers[0]
    result = [max_num]
    for num in numbers[1:]:
        if num > max_num:
            max_num = num
        result.append(max_num)
    return result
stderr: Traceback (most recent call last):
  File "/tmp/tmp145kyuxv.py", line 33, in <module>
    check(rolling_max)
  File "/tmp/tmp145kyuxv.py", line 27, in check
    assert candidate([]) == []
           ^^^^^^^^^^^^^
  File "/tmp/tmp145kyuxv.py", line 10, in rolling_max
    max_num = numbers[0]
              ~~~~~~~^^^
IndexError: list index out of range


HumanEval/10  —  FAIL ✗
Completion:
    return string == string[::-1]
stderr: Traceback (most recent call last):
  File "/tmp/tmp2z3q6irh.py", line 38, in <module>
    check(make_palindrome)
  File "/tmp/tmp2z3q6irh.py", line 31, in check
    assert candidate('') == ''
           ^^^^^^^^^^^^^^^^^^^
AssertionError


HumanEval/26  —  FAIL ✗
Completion:
    count_dict = {}
    result = []
    for num in numbers:
        if